# 05 - Source-perturbation scale

Applies the trained network across maximum source perturbations from field-realistic
(a few metres) to the stress-test scale used in training.

## 1. Generate stacks at each perturbation scale

In [ ]:
# realistic-scale source perturbation (Part A) ---
# Reviewer #2 notes field relocations are ~0.5-3 m, and Isaenkov et al. (2022) used
# controlled shifts of 20-30 m -- versus our +/-200 m. This tests the field-realistic
# range alongside the original stress-test case.
import os
os.makedirs('perturb_2.3/data_cache', exist_ok=True)

PERTURB_LEVELS_M = [3.0, 20.0, 30.0, 50.0, 100.0, 200.0]
perturb_stacks = {}

N_REALIZATIONS = 5   # average over this many random draws per perturbation level

for d_max in PERTURB_LEVELS_M:
    label = f"{d_max:.0f}m"
    for r in range(N_REALIZATIONS):
        torch.manual_seed(GLOBAL_SEED + 1000*r)   # different draw per realization,
                                                    # same set of draws across levels
        data_p, locs_p = run_deepwave_simulation(
            vp_model_data=monitoring_stage1_data, n_shots=N_SHOTS_BAD,
            n_receivers_per_shot=300, max_offset_m=d_max,
            dx=dx, dt=dt, freq=freq, nt=nt, peak_time=peak_time,
            source_depth=source_depth_grid, receiver_depth=receiver_depth_grid,
            first_source=first_source_grid, first_receiver=first_receiver_grid,
            model_name=f'Perturb_{label}_r{r}')
        stack_p = process_and_stack_dataset(
            seismic_data=data_p, dataset_name=f"Perturb_{label}_r{r}",
            source_locations_for_this_data=locs_p,
            save_dir_figures='perturb_2.3/figures', save_dir_numpy='perturb_2.3/stacked')
        np.save(f'perturb_2.3/data_cache/stack_{label}_r{r}.npy', normalize_data(stack_p.T))
        np.save(f'perturb_2.3/data_cache/locs_{label}_r{r}.npy',
                locs_p[:, 0, 0].cpu().numpy() * dx)

np.save('perturb_2.3/data_cache/Y_full.npy', Y_full)
print(f"\nSaved {len(perturb_stacks)} perturbation levels + Y_full.")

## 2. Evaluate

In [ ]:
# evaluate the trained U-Net at each perturbation level ---
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt, json, os
from skimage.metrics import structural_similarity as ssim

WINDOW_SIZE, STEP_SIZE = 128, 5
LEVELS = ["3m", "20m", "30m", "50m", "100m", "200m"]
LEVEL_VALUES = [3, 20, 30, 50, 100, 200]

# (reuse sliding_window_view / reconstruct_from_patches_average / calculate_nrms
#  from the 1.7 cell if they're still defined in this kernel)

MODEL_PATH = 'models/unet_32filters.keras'
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
Y_full = np.load('perturb_2.3/data_cache/Y_full.npy')

results, predictions, inputs = {}, {}, {}
for label in LEVELS:
    X = np.load(f'perturb_2.3/data_cache/stack_{label}.npy')
    inputs[label] = X

    view = sliding_window_view(X, (WINDOW_SIZE, WINDOW_SIZE), STEP_SIZE)
    patches = view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]
    pred = reconstruct_from_patches_average(model.predict(patches, verbose=0),
                                            X.shape, WINDOW_SIZE, STEP_SIZE)
    predictions[label] = pred

    dr = Y_full.max() - Y_full.min()
    results[label] = {
        'input_nrms': calculate_nrms(X, Y_full),
        'input_ssim': ssim(Y_full, X, data_range=dr),
        'unet_nrms':  calculate_nrms(pred, Y_full),
        'unet_ssim':  ssim(Y_full, pred, data_range=dr),
    }

print(f"{'Perturbation':14s} {'Input NRMS':>11s} {'U-Net NRMS':>11s} {'Input SSIM':>11s} {'U-Net SSIM':>11s}")
for label in LEVELS:
    r = results[label]
    print(f"+/-{label:11s} {r['input_nrms']:10.2f}% {r['unet_nrms']:10.2f}% "
          f"{r['input_ssim']:11.4f} {r['unet_ssim']:11.4f}")

with open('perturb_2.3/perturbation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# --- Trend plot, with the field-realistic range shaded ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for ax, key_u, key_i, ylab, title in [
    (ax1, 'unet_nrms', 'input_nrms', 'NRMS (%)', 'NRMS vs. source perturbation'),
    (ax2, 'unet_ssim', 'input_ssim', 'SSIM', 'SSIM vs. source perturbation')]:
    ax.plot(LEVEL_VALUES, [results[l][key_u] for l in LEVELS], 'o-', label='U-Net output')
    ax.plot(LEVEL_VALUES, [results[l][key_i] for l in LEVELS], 's--', label='Unprocessed input')
    ax.axvspan(20, 30, alpha=0.15, color='green',
               label='Isaenkov et al. (2022) range')
    ax.set_xscale('log'); ax.set_xlabel('Max source perturbation (m)')
    ax.set_ylabel(ylab); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
os.makedirs('perturb_2.3/figures', exist_ok=True)
plt.savefig('perturb_2.3/figures/perturbation_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Sections: shot geometry + input + U-Net output at each level ---
fig, axes = plt.subplots(3, len(LEVELS), figsize=(4*len(LEVELS), 12))
vabs = np.percentile(np.abs(Y_full), 99)

for j, label in enumerate(LEVELS):
    shot_x = np.load(f'perturb_2.3/data_cache/locs_{label}.npy')

    ax = axes[0, j]
    ax.scatter(shot_x, np.zeros_like(shot_x), marker='*', s=150,
               color='lime', edgecolor='black')
    for x in shot_x:
        ax.axvline(x, color='lime', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_ylim(-1, 1); ax.set_yticks([])
    ax.set_title(f'+/-{label} perturbation', fontweight='bold')
    if j == 0: ax.set_ylabel('SHOT GEOMETRY', fontweight='bold')

    ax = axes[1, j]
    ax.imshow(inputs[label], cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    ax.set_title(f"Input: NRMS={results[label]['input_nrms']:.1f}%", fontsize=10)
    if j == 0: ax.set_ylabel('STACKED INPUT', fontweight='bold')

    ax = axes[2, j]
    ax.imshow(predictions[label], cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    ax.set_title(f"U-Net: NRMS={results[label]['unet_nrms']:.1f}%", fontsize=10)
    ax.set_xlabel('CMP index')
    if j == 0: ax.set_ylabel('U-NET OUTPUT', fontweight='bold')

plt.suptitle('performance across field-realistic to stress-test perturbation scales',
             fontsize=15)
plt.tight_layout()
plt.savefig('perturb_2.3/figures/perturbation_sections.png', dpi=300, bbox_inches='tight')
plt.show()